In [1]:
import pandas as pd

train = pd.read_csv(
    "data/train.csv",
    parse_dates=["Date"],
    dtype={"StateHoliday": str},
    low_memory=False
)

store = pd.read_csv("data/store.csv")

data = train.merge(store, on="Store", how="left")

cutoff_date = "2015-07-01"

model_train = data[data["Date"] < cutoff_date].copy()
model_test = data[data["Date"] >= cutoff_date].copy()

y_test_true = model_test["Sales"].copy()

model_test_without_sales = model_test.drop(columns=["Sales"])

In [3]:
print(data["Date"].min())
print(data["Date"].max())

2013-01-01 00:00:00
2015-07-31 00:00:00


In [4]:
cutoff_date = "2015-07-01"

model_train = data[data["Date"] < cutoff_date].copy()
model_test = data[data["Date"] >= cutoff_date].copy()

y_test_true = model_test["Sales"].copy()
model_test_without_sales = model_test.drop(columns=["Sales"])

print("Training period:")
print(model_train["Date"].min(), model_train["Date"].max())

print("Hidden test period:")
print(model_test["Date"].min(), model_test["Date"].max())

print(model_train.shape)
print(model_test_without_sales.shape)

Training period:
2013-01-01 00:00:00 2015-06-30 00:00:00
Hidden test period:
2015-07-01 00:00:00 2015-07-31 00:00:00
(982644, 18)
(34565, 17)


In [8]:
print(model_train["Sales"].head())
print(y_test_true.head())
print(model_test_without_sales.columns)


34565     5735
34566     9863
34567    13261
34568    13106
34569     6635
Name: Sales, dtype: int64
0     5263
1     6064
2     8314
3    13995
4     4822
Name: Sales, dtype: int64
Index(['Store', 'DayOfWeek', 'Date', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval'],
      dtype='str')


In [9]:
def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    return np.sqrt(
        np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2)
    )